# 01 · Сбор данных

Этот ноутбук собирает сырые данные проекта.

- Часть 1: почасовой PM2.5 станции из OpenAQ (цепочка «локация → сенсор → измерения»).
- Часть 2: погода и CAMS из Open-Meteo (добавляется отдельно).

Запускать сверху вниз.

## Шаг 1. Импорты

Подключаем pandas и наш модуль `src/collect.py`. Ноутбук запускается из папки `notebooks/`, поэтому добавляем корень проекта в путь.

In [1]:
import os
import sys

import pandas as pd

sys.path.append("..")
from src import collect

RAW_PATH = "../data/raw/openaq_pm25.csv"

## Шаг 2. Что выгружаем

Сенсор PM2.5 и период взяты из задачи 1.1. Время в сырых данных — UTC, как его отдаёт API.

In [2]:
print("sensor_id:", collect.SENSOR_ID)
print("from (UTC):", collect.PERIOD_FROM_UTC)
print("to (UTC):", collect.PERIOD_TO_UTC)
start = pd.Timestamp(collect.PERIOD_FROM_UTC)
end = pd.Timestamp(collect.PERIOD_TO_UTC)
max_hours = int((end - start) / pd.Timedelta(hours=1))
print("max hours in period:", max_hours)

sensor_id: 25903
from (UTC): 2020-04-09T16:00:00Z
to (UTC): 2025-11-14T11:00:00Z
max hours in period: 49075


## Шаг 3. Выгрузка один раз

Качаем страницами по 1000 строк до пустой страницы, по годовым окнам. Между запросами пауза 1 с (лимит API — 60 запросов в минуту), при ответе 429 есть повтор. Сырой файл не перезаписываем: если он уже есть, API не вызываем.

In [3]:
if os.path.exists(RAW_PATH):
    print("Файл уже есть, API не вызываем:", RAW_PATH)
else:
    pm25_download = collect.download_pm25()
    collect.save_raw_csv(pm25_download, RAW_PATH)
    print("Сохранено строк:", len(pm25_download))

Файл уже есть, API не вызываем: ../data/raw/openaq_pm25.csv


## Шаг 4. Читаем сырой файл

Читаем CSV с диска и смотрим размер и первые строки.

In [4]:
pm25_raw = pd.read_csv(RAW_PATH)
print(pm25_raw.shape)
pm25_raw.head()

(37823, 4)


,datetime_from_utc,datetime_to_utc,pm25,has_flags
0,2020-04-09T16:00:00Z,2020-04-09T17:00:00Z,6.0,False
1,2020-04-09T17:00:00Z,2020-04-09T18:00:00Z,10.0,False
2,2020-04-09T18:00:00Z,2020-04-09T19:00:00Z,8.0,False
3,2020-04-09T19:00:00Z,2020-04-09T20:00:00Z,6.0,False
4,2020-04-09T20:00:00Z,2020-04-09T21:00:00Z,4.0,False


## Шаг 5. Инспекция

Смотрим типы колонок и пропуски. Ничего не исправляем: чистка — это Этап 2.

In [5]:
print(pm25_raw.dtypes)
print()
print(pm25_raw.isna().sum())

datetime_from_utc        str
datetime_to_utc          str
pm25                 float64
has_flags               bool
dtype: object

datetime_from_utc     0
datetime_to_utc       0
pm25                 66
has_flags             0
dtype: int64


## Шаг 6. Время

Парсим метку начала часа явно (UTC, формат задан) и считаем NaT: так видно, не сломались ли даты.

In [6]:
times = pd.to_datetime(
    pm25_raw["datetime_from_utc"], utc=True, format="%Y-%m-%dT%H:%M:%SZ", errors="coerce"
)
print("NaT:", times.isna().sum())
print("first:", times.min())
print("last:", times.max())

NaT: 0
first: 2020-04-09 16:00:00+00:00
last: 2025-11-14 10:00:00+00:00


## Шаг 7. Полнота ряда

Сравниваем число строк с максимумом часов в периоде: разница — пропущенные часы (дыры станции). Также считаем дубли меток и строки по годам.

In [7]:
unique_hours = times.nunique()
missing_hours = max_hours - unique_hours
missing_share = missing_hours / max_hours * 100
print("max hours in period:", max_hours)
print("unique hours:", unique_hours)
print("missing hours:", missing_hours)
print("missing share, %:", round(missing_share, 1))
print("duplicated timestamps:", times.duplicated().sum())
print()
print(times.dt.year.value_counts().sort_index())

max hours in period: 49075
unique hours: 37823
missing hours: 11252
missing share, %: 22.9
duplicated timestamps: 0

datetime_from_utc
2020    4735
2021    5124
2022    4728
2023    7344
2024    8318
2025    7574
Name: count, dtype: int64


## Шаг 8. Значения PM2.5

Смотрим распределение значений (µg/m³) и считаем отрицательные значения и нули.

In [8]:
print(pm25_raw["pm25"].describe())
print("negative:", (pm25_raw["pm25"] < 0).sum())
print("zeros:", (pm25_raw["pm25"] == 0).sum())

count    37757.000000
mean        29.547236
std         37.862882
min          0.000000
25%          9.000000
50%         15.000000
75%         31.000000
max        421.000000
Name: pm25, dtype: float64
negative: 0
zeros: 90


## Итог

Сырой файл прочитан и осмотрен без изменений. В Этап 2 передаём то, что видно в выводе ячеек выше: пропущенные часы (дыры ряда), дубли меток времени, если они есть, и отрицательные значения PM2.5, если они есть. Решать, что с ними делать, будем при очистке.